# Train the AC thermal surrogate

This notebook trains the manuscript's AC CNN-LSTM surrogate and
writes `models/thermal/ac_model.pth`. The measured L14 input is intentionally
excluded from the repository; place it at
`data/private/l14_merged_data_with_rain.csv` before execution.


In [ ]:
MODEL_FILENAME = "ac_model.pth"
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import root_mean_squared_error, mean_absolute_percentage_error, mean_absolute_error

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler

from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "models").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from inside the MMV4EF release repository.")


REPO_ROOT = find_repo_root()
DATA_PATH = REPO_ROOT / "data" / "private" / "l14_merged_data_with_rain.csv"
MODEL_PATH = REPO_ROOT / "models" / "thermal" / MODEL_FILENAME


In [ ]:
df = pd.read_csv(DATA_PATH)


In [ ]:
# Filter the dataframe
filtered_df = df[df['Z1 Windows Open Close Status'] == 0]

# Convert the 'date' column to datetime
filtered_df['date'] = pd.to_datetime(filtered_df['date'])

# Filter the dataframe to only include rows where the time is between 7:30 and 19:00
filtered_df = filtered_df[(filtered_df['date'].dt.time >= pd.to_datetime('07:30').time()) & 
                          (filtered_df['date'].dt.time <= pd.to_datetime('19:00').time())]

# Remove weekends from the dataframe
filtered_df = filtered_df[filtered_df['date'].dt.weekday < 5]

# Remove specific date
# dates_to_remove = pd.to_datetime(['2024-10-31', '2024-11-04', '2024-08-09', '2024-07-07'])
dates_to_remove = pd.to_datetime(['2024-10-31', '2024-11-04'])
filtered_df = filtered_df[~filtered_df['date'].dt.normalize().isin(dates_to_remove)]
    
# Sort the dataframe by date
filtered_df = filtered_df.sort_values(by='date')

filtered_df.loc[:, 'Solar Radiation'] = filtered_df['Solar Radiation'].interpolate(method='nearest', limit_direction='both')

# Create a list to store the dataframes
list_of_dfs = []

# Initialize the first dataframe
current_df = [filtered_df.iloc[0]]

# Iterate through the filtered dataframe
for i in range(1, len(filtered_df)):
    # Check if the current date is continuous with the previous date
    if (filtered_df.iloc[i]['date'] - filtered_df.iloc[i-1]['date']).seconds == 60:
        current_df.append(filtered_df.iloc[i])
    else:
        # If not continuous, save the current dataframe and start a new one
        list_of_dfs.append(pd.DataFrame(current_df))
        current_df = [filtered_df.iloc[i]]

# # Append the last dataframe
# if not pd.DataFrame(current_df)[['Zone 1 Temperature', 'Zone 2 Temperature', 'Zone 3 Temperature', 'Zone 4 Temperature', 'Zone 5 Temperature', 'OutdoorTemperatureWindow', 'Wind Speed', 'PFCU-01 Supply Air Temp', 'PFCU-02 Supply Air Temp']].isna().any().any():
#     list_of_dfs.append(pd.DataFrame(current_df))
list_of_dfs.append(pd.DataFrame(current_df))

# Print the number of dataframes
print(f"Number of continuous dataframes: {len(list_of_dfs)}")


In [ ]:
list_of_dfs = [df for df in list_of_dfs if len(df) >= 10]


In [ ]:
# Reset indices in each dataframe in list_of_dfs
list_of_dfs = [df.reset_index(drop=True) for df in list_of_dfs]


In [ ]:
# Define the columns to check for NaN values
columns_to_check = ['Zone 1 Temperature', 'Zone 2 Temperature', 'Zone 3 Temperature', 'Zone 4 Temperature', 'Zone 5 Temperature', 'OutdoorTemperatureWindow', 'FCU-01 Supply Air Temp', 'FCU-02 Supply Air Temp - 1 min', 'FCU-03 Supply Air Temp', 'FCU-04 Supply Air Temp', 'FCU-05 Supply Air Temp', 'PFCU-01 Supply Air Temp', 'PFCU-02 Supply Air Temp','rain_status', 'Solar Radiation']

# Filter the list_of_dfs to drop dataframes with NaN values in the specified columns
list_of_dfs = [df.dropna(subset=columns_to_check) for df in list_of_dfs if not df[columns_to_check].isna().any().any()]

# Print the number of dataframes after filtering
print(f"Number of dataframes after filtering: {len(list_of_dfs)}")


In [ ]:
# Define a CNN-LSTM model
class CNNLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(CNNLSTM, self).__init__()
        self.conv = nn.Conv1d(in_channels=input_dim, out_channels=hidden_dim, kernel_size=1)
        self.lstm = nn.LSTM(input_size=hidden_dim, hidden_size=hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        x = x.transpose(1, 2)              # (batch, input_dim, seq_len)
        x = self.conv(x)                   # (batch, hidden_dim, seq_len)
        x = x.transpose(1, 2)              # (batch, seq_len, hidden_dim)
        out, _ = self.lstm(x)              # (batch, seq_len, hidden_dim)
        out = self.fc(out[:, -1, :])       # last timestep
        return out


In [ ]:
# Define the split ratio
split_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15
# Calculate the split index
split_index = int(len(list_of_dfs) * split_ratio)
val_index = int(len(list_of_dfs) * (split_ratio + val_ratio))

# Split the list of dataframes
train_dfs = list_of_dfs[:split_index]
val_dfs = list_of_dfs[split_index:val_index]
test_dfs = list_of_dfs[val_index:]

# Print the number of dataframes in each set
print(f"Number of training dataframes: {len(train_dfs)}")
print(f"Number of validation dataframes: {len(val_dfs)}")
print(f"Number of test dataframes: {len(test_dfs)}")


In [ ]:
# Initialize scalers
scaler_X = MinMaxScaler()
scaler_U = MinMaxScaler()

# Concatenate all dataframes in train_dfs and val_dfs
all_dfs = pd.concat(train_dfs + val_dfs + test_dfs, ignore_index=True)

# Fit the scalers on the entire dataset
scaler_X.fit(all_dfs[['Zone 1 Temperature', 'Zone 2 Temperature', 'Zone 3 Temperature', 'Zone 4 Temperature', 'Zone 5 Temperature']])
scaler_U.fit(all_dfs[['OutdoorTemperatureWindow', 'FCU-01 Supply Air Temp', 'FCU-02 Supply Air Temp - 1 min', 'FCU-03 Supply Air Temp', 'FCU-04 Supply Air Temp', 'FCU-05 Supply Air Temp', 'PFCU-01 Supply Air Temp', 'PFCU-02 Supply Air Temp','rain_status', 'Solar Radiation']])


In [ ]:
# Function to prepare data for LSTM
def prepare_data(df, scaler_X, scaler_U, discard_trasition=0):
    timestamp = df['date'].values[discard_trasition:]
    X_scaled = scaler_X.transform(df[['Zone 1 Temperature', 'Zone 2 Temperature', 'Zone 3 Temperature', 'Zone 4 Temperature', 'Zone 5 Temperature']])
    U_scaled = scaler_U.transform(df[['OutdoorTemperatureWindow', 'FCU-01 Supply Air Temp', 'FCU-02 Supply Air Temp - 1 min', 'FCU-03 Supply Air Temp', 'FCU-04 Supply Air Temp', 'FCU-05 Supply Air Temp', 'PFCU-01 Supply Air Temp', 'PFCU-02 Supply Air Temp','rain_status', 'Solar Radiation']])
    X_t = X_scaled[discard_trasition:-1]
    X_t1 = X_scaled[discard_trasition+1:]
    U_t = U_scaled[discard_trasition:-1]
    
    input_data = np.hstack([X_t, U_t])
    input_data = input_data.reshape((input_data.shape[0], 1, input_data.shape[1]))
    
    input_data = torch.tensor(input_data, dtype=torch.float32).to(device)
    X_t1 = torch.tensor(X_t1, dtype=torch.float32).to(device)
    return input_data, X_t1, timestamp



# Define model parameters
input_dim = 15
hidden_dim = 64
output_dim = 5
num_epochs = 15


In [ ]:
# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Initialize the model, criterion, and optimizer
model = CNNLSTM(input_dim, hidden_dim, output_dim).to(device)
criterion = nn.SmoothL1Loss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop with validation evaluation
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_batches = 0
    for df in train_dfs:
        input_data_train, X_t1_train, _ = prepare_data(df, scaler_X, scaler_U, discard_trasition=0)
        if torch.isnan(input_data_train).any() or torch.isnan(X_t1_train).any():
            print("NaN values found in the training data, skipping this batch.")
            continue
        outputs = model(input_data_train)
        optimizer.zero_grad()
        loss = criterion(outputs, X_t1_train)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_batches += 1

    model.eval()
    val_loss = 0.0
    val_batches = 0
    with torch.no_grad():
        for df in val_dfs:
            input_data_val, X_t1_val, _ = prepare_data(df, scaler_X, scaler_U, discard_trasition=0)
            if torch.isnan(input_data_val).any() or torch.isnan(X_t1_val).any():
                print("NaN values found in the validation data, skipping this batch.")
                continue
            val_outputs = model(input_data_val)
            val_loss += criterion(val_outputs, X_t1_val).item()
            val_batches += 1

    avg_train_loss = train_loss / train_batches if train_batches else float('nan')
    avg_val_loss = val_loss / val_batches if val_batches else float('nan')

    # if (epoch + 1) % 10 == 0:
    print(f'Epoch [{epoch + 1}/{num_epochs}], '
            f'Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}')


In [ ]:
# Recursive prediction loop
model.eval()
x_t1_val_all = []
x_t1_pred_all = []
for df in test_dfs:
    input_data_val, X_t1_val, timestamp = prepare_data(df, scaler_X, scaler_U, discard_trasition=0)
    with torch.no_grad():
        # Initialize the recursive prediction with the first ground truth value
        print(timestamp[0])
        X_t1_pred_recursive = [input_data_val[0][0,:5]]
        for i in range(1, len(input_data_val)):
            next_X = X_t1_pred_recursive[-1].unsqueeze(0).unsqueeze(0)
            next_U = input_data_val[i][0,5:].unsqueeze(0).unsqueeze(0)
            next_input = torch.cat([next_X, next_U], dim=2)
            next_pred = model(next_input)
            X_t1_pred_recursive.append(next_pred.squeeze(0))
        
        # Convert the list of predictions to a tensor
        X_t1_pred_recursive = torch.cat(X_t1_pred_recursive, dim=0).view(-1, 5)
    # Inverse transform the predictions and ground truth
    input_data_val_inverse = scaler_U.inverse_transform(input_data_val[:,:,5:].squeeze(1).cpu())
    X_t1_val_inverse = scaler_X.inverse_transform(X_t1_val.cpu())
    X_t1_pred_recursive_inverse = scaler_X.inverse_transform(X_t1_pred_recursive.cpu())
    x_t1_val_all.append(X_t1_val_inverse)
    x_t1_pred_all.append(X_t1_pred_recursive_inverse)

    # Calculate metrics
    rmse_val_recursive = root_mean_squared_error(X_t1_val_inverse, X_t1_pred_recursive_inverse)
    mae_val_recursive = mean_absolute_error(X_t1_val_inverse, X_t1_pred_recursive_inverse)
    
    print(f"Recursive Validation RMSE: {rmse_val_recursive}")
    print(f"Recursive Validation MAE: {mae_val_recursive}")


    # Plot comparison for 5 X columns
    plt.figure(figsize=(15, 10))
    for i, col in enumerate(['Zone 1 Temperature', 'Zone 2 Temperature', 'Zone 3 Temperature', 'Zone 4 Temperature', 'Zone 5 Temperature']):
        plt.subplot(3, 2, i + 1)
        plt.plot(X_t1_val_inverse[:, i], label='Actual')
        plt.plot(X_t1_pred_recursive_inverse[1:, i], label='Predicted')
        plt.plot(input_data_val_inverse[:,0], label='OAT')
        plt.title(col)
        plt.legend()
    plt.tight_layout()
    plt.show()

# Calculate overall metrics
overall_mae = mean_absolute_error(np.vstack(x_t1_val_all), np.vstack(x_t1_pred_all))
overall_mape = mean_absolute_percentage_error(np.vstack(x_t1_val_all), np.vstack(x_t1_pred_all))

print(f"Overall Recursive Validation MAE: {overall_mae}")
print(f"Overall Recursive Validation MAPE: {overall_mape}")


In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
model_path = MODEL_PATH
torch.save(
	{
		'model_state_dict': model.state_dict(),
		'input_dim': input_dim,
		'hidden_dim': hidden_dim,
		'output_dim': output_dim,
		'scaler_X': scaler_X,
		'scaler_U': scaler_U,
	},
	model_path,
)
print(f"Model saved to {model_path}")
